In [5]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
url_titanic = 'https://raw.githubusercontent.com/datasciencedojo/datasets/refs/heads/master/titanic.csv'
df_titanic = pd.read_csv(url_titanic)

print("Procent brakujących danych dla każdej cechy:")
print((df_titanic.isna().mean() * 100).round(2))

df_titanic['Embarked'] = df_titanic['Embarked'].fillna('U')

df_t1 = df_titanic.copy()
df_t1 = pd.get_dummies(df_t1, columns=['Sex', 'Embarked'], drop_first=False)

num_cols_1 = df_t1.select_dtypes(include=[np.number]).dropna()

vif_1 = pd.DataFrame()
vif_1["cecha"] = num_cols_1.columns
vif_1["VIF"] = [variance_inflation_factor(num_cols_1.values, i) for i in range(len(num_cols_1.columns))]

print("\n--- VIF (Z pułapką zmiennych zero-jedynkowych) ---")
print(vif_1.sort_values(by="VIF", ascending=False))

df_t2 = df_titanic.copy()
df_t2 = pd.get_dummies(df_t2, columns=['Sex', 'Embarked'], drop_first=True)

num_cols_2 = df_t2.select_dtypes(include=[np.number]).dropna()

vif_2 = pd.DataFrame()
vif_2["cecha"] = num_cols_2.columns
vif_2["VIF"] = [variance_inflation_factor(num_cols_2.values, i) for i in range(len(num_cols_2.columns))]

print("\n--- VIF (BEZ pułapki - drop_first=True) ---")
print(vif_2.sort_values(by="VIF", ascending=False))

Procent brakujących danych dla każdej cechy:
PassengerId     0.00
Survived        0.00
Pclass          0.00
Name            0.00
Sex             0.00
Age            19.87
SibSp           0.00
Parch           0.00
Ticket          0.00
Fare            0.00
Cabin          77.10
Embarked        0.22
dtype: float64

--- VIF (Z pułapką zmiennych zero-jedynkowych) ---
         cecha       VIF
2       Pclass  4.331430
3          Age  3.799999
0  PassengerId  3.705255
6         Fare  1.873504
1     Survived  1.712302
4        SibSp  1.622290
5        Parch  1.554214

--- VIF (BEZ pułapki - drop_first=True) ---
         cecha       VIF
2       Pclass  4.331430
3          Age  3.799999
0  PassengerId  3.705255
6         Fare  1.873504
1     Survived  1.712302
4        SibSp  1.622290
5        Parch  1.554214


In [8]:
import pandas as pd
import category_encoders as ce

url_adult = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

kolumny = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
           'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
           'hours-per-week', 'native-country', 'income']

df_adult = pd.read_csv(url_adult, header=None, names=kolumny, skipinitialspace=True)
df_adult = pd.get_dummies(df_adult, columns=['workclass'], drop_first=True)

ameryka_pn = ['United-States', 'Canada', 'Mexico', 'Puerto-Rico', 'Outlying-US(Guam-USVI-etc)', 'Cuba', 'Jamaica', 'Dominican-Republic', 'Haiti', 'Guatemala', 'Honduras', 'El-Salvador', 'Nicaragua']
azja = ['India', 'Iran', 'Philippines', 'Cambodia', 'Thailand', 'Laos', 'Taiwan', 'China', 'Japan', 'Vietnam', 'Hong']
europa = ['Germany', 'England', 'Italy', 'Poland', 'Portugal', 'France', 'Yugoslavia', 'Scotland', 'Greece', 'Ireland', 'Hungary', 'Holand-Netherlands']
ameryka_pd = ['Columbia', 'Ecuador', 'Peru', 'Trinadad&Tobago']

def przypisz_region(kraj):
    if pd.isna(kraj) or kraj == '?': return 'Nieznany'
    if kraj in ameryka_pn: return 'Ameryka Północna'
    elif kraj in azja: return 'Azja'
    elif kraj in europa: return 'Europa'
    elif kraj in ameryka_pd: return 'Ameryka Południowa'
    else: return 'Inny'

df_adult['region'] = df_adult['native-country'].apply(przypisz_region)
df_adult = pd.get_dummies(df_adult, columns=['region'], drop_first=True)

encoder = ce.BinaryEncoder(cols=['occupation'])
df_adult = encoder.fit_transform(df_adult)
df_adult = pd.get_dummies(df_adult, columns=['race', 'sex'], drop_first=True)

print("Sukces! Podgląd przekształconej ramki (tylko kolumny z Zadań 2.1 - 2.3):")
print(df_adult.filter(regex='workclass|region|occupation|race|sex').head())

Sukces! Podgląd przekształconej ramki (tylko kolumny z Zadań 2.1 - 2.3):
   occupation_0  occupation_1  occupation_2  occupation_3  \
0             0             0             0             1   
1             0             0             1             0   
2             0             0             1             1   
3             0             0             1             1   
4             0             1             0             0   

   workclass_Federal-gov  workclass_Local-gov  workclass_Never-worked  \
0                  False                False                   False   
1                  False                False                   False   
2                  False                False                   False   
3                  False                False                   False   
4                  False                False                   False   

   workclass_Private  workclass_Self-emp-inc  workclass_Self-emp-not-inc  ...  \
0              False                   F